# 🎯 Nesne Tespiti: SSD MobileNet V1 ve EfficientDet-Lite0
Raspberry Pi 5 + LiteRT. Ders notu: `04-object-detection.md`

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from ai_edge_litert.interpreter import Interpreter

## 1) SSD MobileNet V1: giriş ve çıkışları incele

In [ ]:
interpreter = Interpreter(model_path="models/ssd-mobilenet-v1-tflite-default-v1.tflite", num_threads=4)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

inp = input_details[0]
print("GİRİŞ:", inp['shape'], inp['dtype'].__name__, "| quantization:", inp['quantization'])
print(f"\nÇIKIŞ SAYISI: {len(output_details)}")
for i, o in enumerate(output_details):
    print(f"  [{i}] {o['name']:35s} shape={o['shape']}  dtype={o['dtype'].__name__}")

## 2) İlk tespit: ham çıktılar

In [ ]:
labels = [line.strip() for line in open("models/coco_labels.txt")]

orig_img = Image.open("images/cat_dog.jpeg").convert("RGB")
W, H = orig_img.size
print("Orijinal boyut:", W, "x", H)

img = orig_img.resize((300, 300))                          # squash
input_data = np.expand_dims(np.array(img), axis=0)        # (1, 300, 300, 3) uint8
print("Model girişi:", input_data.shape, input_data.dtype)

In [ ]:
interpreter.set_tensor(input_details[0]['index'], input_data)
interpreter.invoke()                                       # warm-up

t0 = time.perf_counter()
interpreter.invoke()
print(f"Çıkarım süresi: {(time.perf_counter() - t0) * 1000:.1f} ms")

boxes   = interpreter.get_tensor(output_details[0]['index'])[0]
classes = interpreter.get_tensor(output_details[1]['index'])[0]
scores  = interpreter.get_tensor(output_details[2]['index'])[0]
count   = int(interpreter.get_tensor(output_details[3]['index'])[0])

print(f"\nModelin döndürdüğü tespit sayısı: {count}\n")
for i in range(count):
    print(f"{i}: {labels[int(classes[i])]:12s} skor={scores[i]:.2f}  kutu={np.round(boxes[i], 3)}")

## 3) Çıkışları içeriğe göre okuyan genel tespit fonksiyonu + çizim

In [ ]:
def read_outputs(itp):
    """Çıkışları SIRAYA değil İÇERİĞE göre tanı: boxes / classes / scores / count."""
    boxes = classes = scores = count = None
    for o in itp.get_output_details():
        t = itp.get_tensor(o['index'])[0] if len(o['shape']) > 1 else itp.get_tensor(o['index'])
        if t.ndim == 2 and t.shape[-1] == 4:
            boxes = t                                   # [N, 4] → kutular
        elif t.size == 1:
            count = int(t.ravel()[0])                   # tek sayı → tespit sayısı
        elif np.allclose(t, np.round(t)):
            classes = t                                 # hep tam sayı → sınıf numaraları
        else:
            scores = t                                  # ondalıklı → skorlar
    return boxes, classes, scores, count


def detect_any(itp, img, threshold=0.5):
    inp = itp.get_input_details()[0]
    h, w = inp['shape'][1], inp['shape'][2]
    x = np.expand_dims(np.array(img.convert("RGB").resize((w, h))), axis=0).astype(inp['dtype'])
    itp.set_tensor(inp['index'], x)
    itp.invoke()                                        # warm-up
    t0 = time.perf_counter()
    itp.invoke()
    ms = (time.perf_counter() - t0) * 1000
    boxes, classes, scores, count = read_outputs(itp)
    dets = [(labels[int(classes[i])], float(scores[i]), np.clip(boxes[i], 0, 1))
            for i in range(count) if scores[i] >= threshold]
    return dets, ms


def draw(img, dets, title=""):
    W, H = img.size
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(img)
    for label, score, (ymin, xmin, ymax, xmax) in dets:
        x, y = xmin * W, ymin * H                      # normalize → piksel
        w, h = (xmax - xmin) * W, (ymax - ymin) * H
        ax.add_patch(patches.Rectangle((x, y), w, h, linewidth=3, edgecolor="red", facecolor="none"))
        ax.text(x, y, f" {label} {score:.0%}", color="white", fontsize=12,
                bbox=dict(facecolor="red", alpha=0.8, pad=2), verticalalignment="bottom")
    ax.set_title(title)
    ax.axis("off")
    plt.show()

In [ ]:
for name in ["cat_dog.jpeg", "home-office.jpg", "office.jpeg"]:
    im = Image.open(f"images/{name}").convert("RGB")
    dets, ms = detect_any(interpreter, im, threshold=0.5)
    draw(im, dets, f"SSD MobileNet V1 | {name} | {len(dets)} nesne | {ms:.1f} ms")

## 4) EfficientDet-Lite0 ile karşılaştırma

In [ ]:
eff = Interpreter(model_path="models/lite-model_efficientdet_lite0_detection_metadata_1.tflite", num_threads=4)
eff.allocate_tensors()
e_in = eff.get_input_details()[0]
print("GİRİŞ:", e_in['shape'], e_in['dtype'].__name__, e_in['quantization'])
for i, o in enumerate(eff.get_output_details()):
    print(f"  [{i}] {o['name']:35s} shape={o['shape']}")

In [ ]:
for name in ["cat_dog.jpeg", "home-office.jpg", "office.jpeg"]:
    im = Image.open(f"images/{name}").convert("RGB")
    for model_name, itp in [("SSD MobileNet V1", interpreter), ("EfficientDet-Lite0", eff)]:
        dets, ms = detect_any(itp, im, threshold=0.5)
        print(f"{name:16s} | {model_name:18s} | {ms:5.1f} ms | " + ", ".join(f"{d[0]} {d[1]:.0%}" for d in dets))
    print()

In [ ]:
im = Image.open("images/home-office.jpg").convert("RGB")
dets, ms = detect_any(eff, im, threshold=0.5)
draw(im, dets, f"EfficientDet-Lite0 | home-office | {len(dets)} nesne | {ms:.1f} ms")